# Lumen Clip — Google Colab GPU backend

One Python process: model + FastAPI + jobs. Free Colab will still disconnect.

**AFTER COLAB RECONNECT:** Cell 3 → Cell 4 → Cell 6 → Cell 8 → Cell 7. Then paste the new PUBLIC API URL.
Skip Cell 5 on recovery. If the runtime was fully reset, Cell 4 must load the model again.


## 1. Install dependencies


In [ ]:
import subprocess, sys
pkgs=['diffusers==0.32.2','transformers==4.46.3','accelerate==1.1.1','safetensors==0.4.5','huggingface_hub==0.26.5','fastapi==0.115.6','uvicorn==0.32.1','imageio==2.36.1','imageio-ffmpeg==0.5.1','opencv-python-headless==4.10.0.84','pydantic==2.10.4']
subprocess.check_call([sys.executable,'-m','pip','install','-q',*pkgs])
print('dependencies ready')


## 2. GPU check


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('PyTorch:', torch.__version__)
print('CUDA version:', torch.version.cuda)
if torch.cuda.is_available():
    print('GPU detected:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2), 'GB')
else:
    raise SystemExit('No GPU. Runtime → Change runtime type → GPU.')


## 3. Download latest GitHub backend


In [ ]:
import sys, urllib.request
from pathlib import Path
REPO='https://raw.githubusercontent.com/sgue19000/t2v-kaggle-webapp/main/colab'
BACKEND_DIR=Path('/content/t2v_backend')
BACKEND_DIR.mkdir(parents=True, exist_ok=True)
for name in ('generator.py','server.py','start_api.py'):
    dest=BACKEND_DIR/name
    urllib.request.urlretrieve(f'{REPO}/{name}', dest)
    print('updated', dest, dest.stat().st_size, 'bytes')
assert (BACKEND_DIR/'server.py').exists()
assert (BACKEND_DIR/'generator.py').exists()
assert (BACKEND_DIR/'start_api.py').exists()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
print('Backend files ready:', BACKEND_DIR)


## 4. Load the video model


In [ ]:
import sys
from pathlib import Path
BACKEND_DIR=Path('/content/t2v_backend')
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
from generator import load_pipeline, model_info, gpu_report
report=gpu_report()
print('GPU detected:', report.get('gpu'))
info=model_info()
if info.get('loaded'):
    print('Model loaded')
    print(info)
else:
    load_pipeline()
    print(model_info())


## 5. Tiny generation test (skip after reconnect)


In [ ]:
import sys
from pathlib import Path
from IPython.display import Video, display
BACKEND_DIR=Path('/content/t2v_backend')
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
from generator import generate_video_with_fallback
demo=Path('/content/outputs/demo.mp4')
path, meta = generate_video_with_fallback({'prompt':'A golden retriever running through tall grass at sunrise','num_frames':8,'height':256,'width':256,'fps':8,'steps':12,'guidance_scale':9,'seed':42}, out_path=demo)
print(meta)
print('wrote', path, 'bytes', path.stat().st_size)
assert path.exists() and path.stat().st_size>1024
display(Video(str(path), embed=True))


## 6. Start FastAPI


In [ ]:
from pathlib import Path
FORCE_RESTART = False
starter = Path('/content/t2v_backend/start_api.py')
assert starter.exists(), 'Re-run Cell 3 to download start_api.py'
exec(compile(starter.read_text(), str(starter), 'exec'), globals())


## 7. Cloudflare Quick Tunnel


In [ ]:
import time, subprocess, re, json, urllib.request
from pathlib import Path
try:
    local=json.load(urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5))
except Exception:
    local=None
if not local or not local.get('ok'):
    raise SystemExit('FastAPI is not healthy on :8000. Re-run Cell 3 and Cell 6.')
print('local /health HTTP 200, loaded=', local.get('loaded'), 'gpu=', local.get('gpu'))
FORCE_NEW_TUNNEL=False
old_url=globals().get('PUBLIC_API_URL')
old_proc=globals().get('T2V_TUNNEL_PROC')
alive=old_proc is not None and old_proc.poll() is None
if old_url and alive and not FORCE_NEW_TUNNEL:
    print('PUBLIC API URL:', old_url)
else:
    cf=Path('/content/cloudflared')
    if not cf.exists():
        subprocess.check_call(['wget','-q','-O',str(cf),'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'])
        cf.chmod(0o755)
    if old_proc is not None and old_proc.poll() is None:
        old_proc.terminate(); time.sleep(1)
    log_path=Path('/content/tunnel.log')
    log=open(log_path,'w')
    proc=subprocess.Popen([str(cf),'tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'], stdout=log, stderr=subprocess.STDOUT)
    globals()['T2V_TUNNEL_PROC']=proc
    url=None
    for _ in range(45):
        time.sleep(1)
        found=re.findall(r'https://[-a-z0-9.]+trycloudflare.com', log_path.read_text(errors='ignore'))
        if found:
            url=found[-1]; break
    if not url:
        raise SystemExit('Tunnel URL missing. Set FORCE_NEW_TUNNEL=True and re-run.')
    print('PUBLIC API URL:', url)
    globals()['PUBLIC_API_URL']=url


## 8. Test /health


In [ ]:
import json, urllib.request
data=json.load(urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=30))
print(json.dumps(data, indent=2))
assert data.get('ok') is True
print('gpu=', data.get('gpu'), 'loaded=', data.get('loaded'), 'server=', data.get('server_status'), 'busy=', data.get('busy'))


## 9. Test /generate


In [ ]:
import json, time, urllib.request
req=urllib.request.Request('http://127.0.0.1:8000/generate', data=json.dumps({'prompt':'A cinematic futuristic city at night, flying cars, rain','num_frames':8,'width':256,'height':256,'fps':8,'steps':12,'guidance_scale':9,'seed':12345}).encode(), headers={'Content-Type':'application/json'}, method='POST')
started=json.load(urllib.request.urlopen(req, timeout=30))
print(started)
job_id=started['job_id']
for _ in range(120):
    snap=json.load(urllib.request.urlopen(f'http://127.0.0.1:8000/status/{job_id}', timeout=30))
    print(snap.get('status'), snap.get('progress'), snap.get('message') or snap.get('error'))
    if snap.get('status') in ('completed','failed'): break
    time.sleep(5)
globals()['LAST_JOB_ID']=job_id


## 10. Display generated MP4


In [ ]:
from IPython.display import Video, display
from pathlib import Path
job_id=globals().get('LAST_JOB_ID')
paths=[Path(f'/content/outputs/{job_id}.mp4')] if job_id else []
paths.append(Path('/content/outputs/demo.mp4'))
shown=False
for p in paths:
    if p.exists() and p.stat().st_size>0:
        print(p, p.stat().st_size); display(Video(str(p), embed=True)); shown=True; break
if not shown: print('No MP4 found')
